# Computation of shape during growth of propagules (2)


##  Documentation and Imports


Created on 27-05-2021 (Adapted from ShapeComputation.ipynb)

Author: Valentin Laplaud

This code aims to compute the shape of propagules growing in microfluidic chips. It detects gemmae contours using binarization, and saves the data of the contour position on the image, as well as the quality of image focus and the position of the center of the gemma on the image.

In [ ]:
# plotting stuff
import matplotlib as mpl
mpl.use('TkAgg')
%matplotlib inline

COLOR = 'white'
COLOR2 = 'black'

mpl.rcParams['text.color'] = COLOR
mpl.rcParams['axes.labelcolor'] = COLOR
mpl.rcParams['xtick.color'] = COLOR
mpl.rcParams['ytick.color'] = COLOR
mpl.rcParams['axes.edgecolor'] = COLOR

mpl.rcParams["figure.facecolor"] = COLOR2
mpl.rcParams["axes.facecolor"] = COLOR2
mpl.rcParams["savefig.facecolor"] = COLOR2
mpl.rcParams['axes.facecolor'] = COLOR2

import matplotlib.pyplot as plt
from cycler import cycler
#Default colors
colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/30) for i in range(30)]
mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)

# numbers handling
import numpy as np
import pandas as pd

# signal processing 
from scipy.signal import savgol_filter, correlate, correlation_lags
from scipy.interpolate import interp1d

# images handling
from skimage import io
from skimage.filters import threshold_otsu, threshold_mean, gaussian, laplace, sobel
from skimage.measure import label, regionprops, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing, remove_small_holes,binary_erosion
from skimage.color import rgb2gray, rgb2hsv
from skimage.segmentation import active_contour, morphological_geodesic_active_contour,morphological_chan_vese, checkerboard_level_set, inverse_gaussian_gradient
import cv2 as cv

# to hide known warnings
import warnings
warnings.filterwarnings("ignore")

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf


##  Define contour computation functions



###  Image binarization

Binarize the image to get a large mask that will be used as a starting point for active contour algorithm

GetHue : Computes Hue from RGB image. Also applies a gaussian blur for smoothness.

Binarize : Threshold the Hue image using the Otsu threshold. Then opens the image using a circle of 40 µm.

create_circular_mask : create the circle for opening

In [ ]:
def GetHue(RGBimg, **kwargs):

    #init and read kwargs
    DebugPlots = False
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        else:
            print('Unknown key : ' + key + '. Kwarg ignored.')
    
    warnings.filterwarnings("ignore")   

    HueImg = np.zeros(RGBimg.shape[0:2]) # initialize Hue img

    Maxes = RGBimg.argmax(axis=2)
    Mines = RGBimg.argmin(axis=2)

    # when red is max
    mask = Maxes == 0
    HueImg[mask] = (RGBimg[mask,1]-RGBimg[mask,2])/(RGBimg[mask,:].max(axis=1)-RGBimg[mask,:].min(axis=1))

    # when green is max
    mask = Maxes == 1
    HueImg[mask] = 2 + (RGBimg[mask,2]-RGBimg[mask,0])/(RGBimg[mask,:].max(axis=1)-RGBimg[mask,:].min(axis=1))

    # when blue is max
    mask = Maxes == 2
    HueImg[mask] = 4 + (RGBimg[mask,0]-RGBimg[mask,1])/(RGBimg[mask,:].max(axis=1)-RGBimg[mask,:].min(axis=1))


    # correct for nans
    HueImg[np.isnan(HueImg)] = 5

    HueImg = (HueImg/6)%1
          
    if DebugPlots:
                
        fig, (ax0, ax1) = plt.subplots(ncols=2, dpi=200)
        
        fig.suptitle('Debug plots for "GetHue" function :')
        ax0.imshow(RGBimg)
        ax0.set_title("RGB image")
        ax0.axis('off')
        ax1.imshow(HueImg, cmap='gray')
        ax1.set_title("Hue image")
        ax1.axis('off')
        
        plt.show()
        
        print('')
    
    return(HueImg)

def GetSat(RGBimg):
    
    Maxes = RGBimg.max(axis=2)
    Mines = RGBimg.min(axis=2)
    
    Int = np.divide(Maxes + Mines,2)
    
    Delta = Maxes-Mines
    
    SatImg = np.divide(Delta,(1-np.abs(2*Int-1)))
    
    return(SatImg)
    
def Binarize(Img, Scale, **kwargs):

    #init and read kwargs
    DebugPlots = False
    BackGrnd = 'white'
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        elif key == 'bckgrnd':
            if (value == 'white') | (value == 'black'):
                BackGrnd = value
    
    
    if BackGrnd == 'black':
        BWimg = Img > threshold_otsu(Img) # apply threshold for black background images
    elif BackGrnd == 'white':
        BWimg = Img > threshold_mean(Img) # apply threshold for white background images
    
    # to avoid filling up to the edges
    BWimg[0:9,0:] = 0    
    BWimg[0:,0:9] = 0 
    BWimg[0:,BWimg.shape[1]-10:BWimg.shape[1]]= 0
    BWimg[BWimg.shape[0]-10:BWimg.shape[0],0:] = 0
    
    FilledBWimg = remove_small_holes(BWimg, area_threshold=1e5) # fills dark regions
    
    Size = np.round(40*Scale) # 40µm in pixels
    
    selem = create_circular_mask(Size,Size) # create circular element for opening

    FinalImg = binary_opening(FilledBWimg,selem) # image opening
    
    if DebugPlots:        
       
        fig, [[ax0, ax1], [ax2, ax3]] = plt.subplots(nrows = 2, ncols=2,dpi = 200)    
        fig.suptitle('Debug plots for "Binarize" function :')
        ax0.imshow(Img, cmap='gray')
        ax0.set_title("Original image")
        ax0.axis('off')
        ax1.imshow(BWimg, cmap='gray')
        ax1.set_title("Thresholded image")
        ax1.axis('off')
        ax2.imshow(FilledBWimg, cmap='gray')
        ax2.set_title("Filled image")
        ax2.axis('off') 
        ax3.imshow(FinalImg, cmap='gray')
        ax3.set_title("Opened image")
        ax3.axis('off')        
        
        plt.show()
        
        print('')
    
    return(FinalImg)

def create_circular_mask(h, w, center=None, radius=None):

    if center is None: # use the middle of the image
        center = (int(w/2), int(h/2))
    if radius is None: # use the smallest distance between the center and image walls
        radius = min(center[0], center[1], w-center[0], h-center[1])

    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y-center[1])**2)

    mask = dist_from_center <= radius
    return mask

### Edge detection from the binary image

Returns the edge positions dnterd around the origin, in cartesian and circular corrdinate. Also returns position of the center in the original image and the area of the propagule

In [ ]:
def getEdgeAndArea(BinImg,Scale):
    
    # Find largest contour 
    cnts, _ = cv.findContours(BinImg, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
    cnt = max(cnts, key=cv.contourArea)
    
    Area = cv.contourArea(cnt)*Scale**2

    # Create image with only largest object
    out = np.zeros(BinImg.shape, np.uint8)
    cv.drawContours(out, [cnt], -1, 255, cv.FILLED)
    BinImg = cv.bitwise_and(BinImg, out)

    # Detecting edges and computing their position
    edge = sobel(BinImg)>threshold_otsu(sobel(BinImg))
    indices = np.where(edge != [0])
    
    # Computing center of propagule using euclidian distance transform
    dist = cv.distanceTransform(BinImg, cv.DIST_L2, 3)
    regions = regionprops(BinImg,dist)
    center = regions[0].weighted_centroid
    
    # Edge coorinate relative to the center
    relativeIndicesX = indices[0]-center[0]
    relativeIndicesY = indices[1]-center[1]
    
    # Edge coordinate in circular system
    Theta,Radius = vf.ToCirc(relativeIndicesX,relativeIndicesY, angle = 'deg')

    # Sorting coordinate based on angle
    FullCoordinates = np.stack((Theta,Radius,relativeIndicesX,relativeIndicesY),axis=1)
    SortedCoord = FullCoordinates[FullCoordinates[:,0].argsort(),]
    
    SortedTheta = SortedCoord[:,0]
    SortedRadius = SortedCoord[:,1]
    SortedEdgeX = SortedCoord[:,2]
    SortedEdgeY = SortedCoord[:,3]
    
    return(SortedEdgeX,SortedEdgeY,center,Area)

##  Main functions
   

### Image Binarization

In [ ]:
def BinarizeImage(StackList, P, Scale, **kwargs):

    #init and read kwargs
    DebugPlots = False
    BackGrnd = 'white'
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        elif key == 'bckgrnd':
            if (value == 'white') | (value == 'black'):
                BackGrnd = value
            else:
                print('Wrong value : ' + value + ' for key : ' + key + '. Using default value.')                    
        else:
            print('Unknown key : ' + key + '. Kwarg ignored.')
    
    for s in StackList:

        print('Processing ' + s + ' :')    

        RGBstack = io.imread(P + '\\' + s + '.tif') # get the tiff stack

        # Savefolders for Sat and Binary stacks
        if not os.path.exists(P + '\\Processed'):
            os.mkdir(P + '\\Processed') # create global folder        
        
        if not os.path.exists(P + '\\Processed\\' + s + '_PreProc'):
            # shutil.rmtree(P + '\\' + s + '_PreProc') # remove folder and contents
            os.mkdir(P + '\\Processed\\' + s + '_PreProc') # create Sat image folder
            isPP = False
        else:
            isPP = True

        if not os.path.exists(P + '\\Processed\\' + s + '_Binarized'):
            # shutil.rmtree(P + '\\' + s + '_Binarized') # remove folder and contents
            os.mkdir(P + '\\Processed\\' + s + '_Binarized') # create binarize image folder
            isBin = False
        else:
            isBin = True


        # pre-process images
        for i in range(len(RGBstack)):
          
            # Sat channel computation
            if not isPP:
                print('Pre-processing image ' + str(i+1) + '/' + str(len(RGBstack)).ljust(10), flush=True, end = '\r')
                if BackGrnd == 'white':
                    PPImg =  rgb2hsv(RGBstack[i])[:,:,1]
                else:
                    PPImg =  rgb2hsv(RGBstack[i])[:,:,0]
                    
                io.imsave(P + '\\Processed\\' + s + '_PreProc\\' + str(i) + '.tif', np.uint8(PPImg*255), plugin='tifffile')

            else:
                if os.path.exists(P + '\\Processed\\' + s + '_PreProc\\' + str(i) + '.tif'):
                    
                    PPImg = io.imread(P + '\\Processed\\' + s + '_PreProc\\' + str(i) + '.tif')
                    
                else:
                    print('Computing Sat for image ' + str(i+1) + '/' + str(len(RGBstack)).ljust(10), flush=True, end = '\r')
                    if BackGrnd == 'white':
                        PPImg =  rgb2hsv(RGBstack[i])[:,:,1]
                    else:
                        PPImg =  rgb2hsv(RGBstack[i])[:,:,0]
                    
                    io.imsave(P + '\\Processed\\' + s + '_PreProc\\' + str(i) + '.tif', np.uint8(PPImg*255), plugin='tifffile')

            # Binarization      
            if not isBin:
                print('Binarization of image ' + str(i+1) + '/' + str(len(RGBstack)).ljust(15), flush=True, end = '\r')
                if BackGrnd == 'black':
                    PPImg = np.invert(PPImg)
                BinImg = Binarize(gaussian(PPImg,2),Scale,debug =DebugPlots, bckgrnd=BackGrnd)
                
                io.imsave(P + '\\Processed\\' + s + '_Binarized\\' + str(i) + '.tif', np.uint8(BinImg*255), plugin='tifffile')
            
            else:
                if os.path.exists(P + '\\Processed\\' + s + '_Binarized\\' + str(i) + '.tif'):
                    BinImg = io.imread(P + '\\Processed\\' + s + '_Binarized\\' + str(i) + '.tif') 
                else:
                    print('Binarization of image ' + str(i+1) + '/' + str(len(RGBstack)).ljust(15), flush=True, end = '\r')
                    if BackGrnd == 'black':
                        PPImg = np.invert(PPImg)
                    BinImg = Binarize(gaussian(PPImg,2),Scale,debug =DebugPlots, bckgrnd=BackGrnd)
                    io.imsave(P + '\\Processed\\' + s + '_Binarized\\' + str(i) + '.tif', np.uint8(BinImg*255), plugin='tifffile')
            


        print('Binarization done for ' + str(len(RGBstack)) + '/' + str(len(RGBstack)) + ' images'.ljust(20), flush=True)

        
    return

### Contour detection

In [ ]:
def GetContours(StackList,P, Scale, FPH, date, **kwargs):
      
    # init and read kwargs    
    DebugPlots = False
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        else:
            print('Unknown key : ' + key + '. Kwarg ignored.')
    
    # initialize dataframme
    GD = pd.DataFrame(data=None,columns=['Img','Time (min)','FocusValue','GoodFocus','Area','Xcenter','Ycenter']) 

    CD = pd.DataFrame(data=None,columns=['Img','Ximg','Yimg','GoodFocus']) 

    
    for s in StackList:

        print('Processing ' + s + ' :')   
        
        # Loading video
        RGBstack = io.imread(P + '\\' + s + '.tif') # get the tiff stack
        
        n = len(RGBstack)
               
        # Evaluating focus quality
        for i in range(n):
            
            print('Checking focus for image ' + str(i+1) + '/' + str(len(RGBstack)).ljust(10), flush=True, end = '\r')
                    
            FocusValue = np.sum(np.absolute(laplace(np.max(RGBstack[i],2))))
            
            GD = GD.append(pd.DataFrame(data = [[i, i/FPH*60, FocusValue]], index = [s], columns = ['Img','Time (min)','FocusValue']))
            
        
        GoodFocusList = np.argwhere(np.concatenate(([0],np.diff(GD.at[s,'FocusValue'])))/GD.at[s,'FocusValue'].median()>-0.05)

        GD.at[s, 'GoodFocus'] = GD.loc[s].apply(
        lambda row: np.isin(row['Img'],GoodFocusList) , axis = 1)

        print('Focus checked.'.ljust(35), flush = True)        
        
        jet_colorcycle = [plt.get_cmap('jet')(1. * i/n) for i in range(n)]
        mpl.rcParams['axes.prop_cycle'] = cycler(color=jet_colorcycle)        


        for i in range(n):
            
            print('Measuring contour for image ' + str(i+1) + '/' + str(len(RGBstack)).ljust(10), flush=True, end = '\r')
                        
            # Loading binary image
            BinImg = io.imread(P + '\\Processed\\' + s + '_Binarized\\' + str(i) + '.tif')

            # Computing propagule edge and area from binary image
            SortedX,SortedY,center,Area = getEdgeAndArea(BinImg,Scale) 
            
            # Storing contour data
            data = {'Ximg':SortedY+center[1],
                        'Yimg':SortedX+center[0],
                        'Img':i*np.ones(len(SortedX)),
                        'GoodFocus':np.repeat(GD.loc[(GD.index == s) &
                                                             (GD['Img'] == i) ,'GoodFocus'],len(SortedX))} 

            CD = CD.append(pd.DataFrame(data=data,index = np.repeat(s,len(SortedX))) ) # adding to global dataframe
            
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Area'] = Area  
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Xcenter'] = center[1] 
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Ycenter'] = center[0]    
            
            if (i == 1) & DebugPlots:
                RGBimg = io.imread(P + '\\' + s + '.tif', key = 1)

                plt.figure(dpi=250,facecolor='white')
                plt.title(s)
                plt.imshow(RGBimg)
                plt.plot(SortedY+center[1],SortedX+center[0],'c',lw = 0.7)
                plt.show()
                
        print('Contours saved.'.ljust(35), flush = True)
         
    return(CD,GD)

## Enter data and run



### Data details

In [ ]:
Scale210318 = 2.27 # Spatial scale (µm/px) 18-03-2021
FPH210318 = 2 # Frames per hour

Scale210319 = 1.72 # Spatial scale (µm/px) 19-03-2021
FPH210319 = 2 # Frames per hour

Scale210528 = 1.94 # Spatial scale (µm/px) 28-05-2021
FPH210528 = 120 # Frames per hour

Scale210618 = 1.05 # Spatial scale (µm/px) 18-06-2021
FPH210618 = 2 # Frames per hour



# Paths to data
P210528 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.05.28'
P210318 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.03.18'
P210319 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.03.19'
P210618 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.06.18'


# experiments of 18-03-2021 (24hours, image every 30 minutes, brightfield, black background, no Rhizoid)
StackList210318 = ['PPG2','PPG3','PPG6','PPG7','PPG9','PPG10','PPG11',
                   'PPG14','PPG15','PPG16','PPG18','PPG19',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']  


# experiments of 19-03-2021 (68h, cropped when air arrives, at least 52 images)
StackList210319 = ['PPG1','PPG2','PPG6','PPG8','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22']



#StackList210319 = ['PPG17','PPG1'] # Tests


# experiments of 28-05-2021 (10 minutes, img every 30s. Test whitefield + control no mouvement)
StackList210528 = ['PPG1','PPG2','PPG3','PPG4','PPG5','PPG6','PPG7','PPG8'
                   ,'PPG9','PPG10','PPG11','PPG12','PPG13','PPG14']
#StackList210528 = ['PPG6']


# experiments of 18-06-2021 (50h, white background test, 1 PPG without flux)

StackList210618 = ['PPG1'] 



print('Data choice made.')


### Run

In [ ]:
## Image Binarization
#BinarizeImage(StackList210318, P210318, Scale210318, debug = False, bckgrnd = 'black')

#BinarizeImage(StackList210319, P210319, Scale210319, debug = False, bckgrnd = 'black')

BinarizeImage(StackList210528, P210528, Scale210528, debug = False, bckgrnd = 'white')

BinarizeImage(StackList210618, P210618, Scale210618, debug = True, bckgrnd = 'white')

In [ ]:
## Contour computation

#ContourData210318,GlobalData210318 = GetContours(StackList210318,P210318, Scale210318, FPH210318 ,'210318', debug=False)

#ContourData210319,GlobalData210319 = GetContours(StackList210319,P210319, Scale210319, FPH210319 ,'210319', debug=False)

ContourData210528,GlobalData210528 = GetContours(StackList210528,P210528, Scale210528, FPH210528 ,'210528', debug=False)

ContourData210618,GlobalData210618 = GetContours(StackList210618,P210618, Scale210618, FPH210618 ,'210618', debug=True)


In [ ]:
## Saving

#GlobalData210318.to_csv(P210318 + '\\GlobalData210318_Img.csv',index_label = 'Ind')
#ContourData210318.to_csv(P210318 + '\\ContourData210318_Img.csv',index_label = 'Ind')

#GlobalData210319.to_csv(P210319 + '\\GlobalData210319_Img.csv',index_label = 'Ind')
#ContourData210319.to_csv(P210319 + '\\ContourData210319_Img.csv',index_label = 'Ind')

GlobalData210528.to_csv(P210528 + '\\GlobalData210528_Img.csv',index_label = 'Ind')
ContourData210528.to_csv(P210528 + '\\ContourData210528_Img.csv',index_label = 'Ind')

GlobalData210618.to_csv(P210618 + '\\GlobalData210618_Img.csv',index_label = 'Ind')
ContourData210618.to_csv(P210618 + '\\ContourData210618_Img.csv',index_label = 'Ind')

# Test Zone

In [ ]:
GlobalData210618

In [ ]:
ContourData210618